# RetailInsight — Exploratory Data Analysis & Visualization
**SRM Institute of Science and Technology | Mini Project DS AIML-B 2026**

**Owner:** Teammate B  
**Objective:** Produce all EDA visualisations from the cleaned dataset — revenue trends, product analysis, customer geography, seasonal patterns, and word clouds.

**Pre-requisite:** Run `preprocessing.ipynb` or `src/preprocessing.py` first.

---

## 1. Setup & Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from wordcloud import WordCloud
import os, warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
os.makedirs('../outputs/graphs', exist_ok=True)

df = pd.read_csv('../dataset/processed_data/retail_clean.csv', parse_dates=['InvoiceDate'])
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(3)

## 2. Monthly Revenue Trend

A time-series line chart showing total monthly revenue across the full dataset period. This reveals seasonality and growth trends.

In [ ]:
monthly = df.groupby('YearMonth')['TotalPrice'].sum().reset_index()
monthly = monthly.sort_values('YearMonth')

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly['YearMonth'], monthly['TotalPrice'], color='#f0a500', linewidth=2.2, marker='o', markersize=4)
ax.fill_between(monthly['YearMonth'], monthly['TotalPrice'], alpha=0.15, color='#f0a500')
ax.set_title('Monthly Revenue Trend (2009–2011)', fontsize=14, pad=12)
ax.set_xlabel('Month')
ax.set_ylabel('Total Revenue (£)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e3:.0f}K'))
plt.xticks(rotation=45, ha='right', fontsize=8)
ax.grid(color='#30363d', linestyle='--', linewidth=0.5, alpha=0.7)
plt.tight_layout()
plt.savefig('../outputs/graphs/monthly_revenue_trend.png', dpi=130, bbox_inches='tight')
plt.show()

## 3. Top 10 Products by Revenue

In [ ]:
top_products = (
    df.groupby('Description')['TotalPrice']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(11, 6))
colors = plt.cm.YlOrRd(np.linspace(0.4, 0.95, len(top_products)))
bars = ax.barh(top_products['Description'].str.title()[::-1], top_products['TotalPrice'][::-1], color=colors, edgecolor='none', height=0.65)

for bar, val in zip(bars, top_products['TotalPrice'][::-1]):
    ax.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2, f'£{val:,.0f}', va='center', fontsize=9, color='#c9d1d9')

ax.set_title('Top 10 Products by Total Revenue', fontsize=13, pad=12)
ax.set_xlabel('Total Revenue (£)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e3:.0f}K'))
ax.grid(axis='x', color='#30363d', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.savefig('../outputs/graphs/top10_products_revenue.png', dpi=130, bbox_inches='tight')
plt.show()

## 4. Revenue by Country (ex-UK)

The UK dominates (~89% of revenue), so we exclude it to clearly see international distribution.

In [ ]:
intl = df[df['Country'] != 'United Kingdom']
country_rev = intl.groupby('Country')['TotalPrice'].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 5))
country_rev.plot(kind='bar', ax=ax, color='#4fc3f7', edgecolor='none')
ax.set_title('Top 10 International Markets by Revenue (ex-UK)', fontsize=13, pad=10)
ax.set_xlabel('Country')
ax.set_ylabel('Revenue (£)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e3:.0f}K'))
plt.xticks(rotation=30, ha='right')
ax.grid(axis='y', color='#30363d', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.savefig('../outputs/graphs/international_revenue.png', dpi=130, bbox_inches='tight')
plt.show()

## 5. Day-of-Week and Hour-of-Day Patterns

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_rev   = df.groupby('DayOfWeek')['TotalPrice'].sum().reindex(day_order)
hour_rev  = df.groupby('Hour')['TotalPrice'].sum()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.bar(day_rev.index, day_rev.values, color='#81c784', edgecolor='none')
ax1.set_title('Revenue by Day of Week', fontsize=12)
ax1.set_ylabel('Revenue (£)')
ax1.tick_params(axis='x', rotation=30)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e3:.0f}K'))
ax1.grid(axis='y', color='#30363d', linestyle='--', linewidth=0.5)

ax2.plot(hour_rev.index, hour_rev.values, color='#e91e63', linewidth=2.2, marker='o', markersize=5)
ax2.fill_between(hour_rev.index, hour_rev.values, alpha=0.15, color='#e91e63')
ax2.set_title('Revenue by Hour of Day', fontsize=12)
ax2.set_xlabel('Hour')
ax2.set_ylabel('Revenue (£)')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e3:.0f}K'))
ax2.grid(color='#30363d', linestyle='--', linewidth=0.5)

plt.tight_layout()
plt.savefig('../outputs/graphs/day_hour_patterns.png', dpi=130, bbox_inches='tight')
plt.show()

## 6. Monthly Transaction Heatmap (Year × Month)

In [ ]:
heatmap_data = (
    df.groupby(['Year', 'Month'])['Invoice']
    .nunique()
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(13, 4))
sns.heatmap(
    heatmap_data, annot=True, fmt='d', cmap='YlOrRd',
    linewidths=0.5, linecolor='#0f1117', ax=ax,
    cbar_kws={'label': 'Unique Invoices'}
)
ax.set_title('Transaction Volume Heatmap (Year × Month)', fontsize=13, pad=10)
ax.set_xlabel('Month')
ax.set_ylabel('Year')
plt.tight_layout()
plt.savefig('../outputs/graphs/transaction_heatmap.png', dpi=130, bbox_inches='tight')
plt.show()

## 7. Product Word Cloud

In [ ]:
text = ' '.join(df['Description'].dropna().values)
# Exclude generic stopwords
stopwords = {'SET', 'OF', 'AND', 'IN', 'THE', 'WITH', 'TO', 'A', 'FOR', 'IS', 'AT', ''}

wc = WordCloud(
    width=1200, height=600,
    background_color='#0f1117',
    colormap='YlOrRd',
    stopwords=stopwords,
    max_words=120,
    collocations=False
).generate(text)

fig, ax = plt.subplots(figsize=(14, 7))
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')
ax.set_title('Most Frequent Product Keywords', fontsize=14, pad=12, color='white')
plt.tight_layout()
plt.savefig('../outputs/graphs/product_wordcloud.png', dpi=130, bbox_inches='tight', facecolor='#0f1117')
plt.show()

## 8. Correlation Heatmap (Numeric Features)

In [ ]:
num_cols = ['Quantity', 'Price', 'TotalPrice', 'Month', 'Hour']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, linecolor='#0f1117', ax=ax, square=True,
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Heatmap', fontsize=12, pad=10)
plt.tight_layout()
plt.savefig('../outputs/graphs/correlation_heatmap.png', dpi=130, bbox_inches='tight')
plt.show()

## Summary

All EDA visualisations saved to `../outputs/graphs/`. Key findings:
- Revenue spikes in **November–December** each year (holiday season)
- **Thursday** is the busiest trading day; **Sunday** the quietest
- Peak purchasing hours: **10 AM – 3 PM**
- **Netherlands, Germany, EIRE** are the largest international markets
- Dominant product categories: giftware, home decor, bags, kitchenware

**Next:** Open `src/analysis.py` for association rule mining or `src/model.py` for clustering.